In [ ]:
# !pip install pyspark
from pyspark import SparkContext
from pyspark.sql import SparkSession
ss = SparkSession.builder \
    .master("local[*]") \
    .appName("outlier detection") \
    .getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
input_folder_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_finale.csv"

df = ss.read.csv(
    input_folder_path,
    header=True,
    inferSchema=True,
    sep=","
)

In [ ]:
df.show()

In [ ]:
from pyspark.sql.functions import col, avg, stddev, lit, abs as spark_abs, greatest

feature_cols = [
    "NDVI", "FDI", "NDWI", "NRD", "NDMI", "BSI", "CON", "DIS", "ENER", "COR", "PC_1"
]

agg_stats = df.agg(*[avg(c).alias(f"mean_{c}") for c in feature_cols],
    *[stddev(c).alias(f"stddev_{c}") for c in feature_cols]).collect()[0]

In [ ]:
zscore_cols = []
zscore_exprs = []

for feature in feature_cols:
    mean_val = agg_stats[f"mean_{feature}"]
    stddev_val = agg_stats[f"stddev_{feature}"]
    zscore_col_name = f"zscore_{feature}"

    zscore_cols.append(zscore_col_name)

    # Compute Z = (X - mu) / sigma
    if stddev_val == 0 or stddev_val is None:
        # If devstd=0 (constant column), z-score=0
        zscore_exprs.append(lit(0.0).alias(zscore_col_name))
    else:
        zscore_exprs.append(
            ((col(feature) - lit(mean_val)) / lit(stddev_val)).alias(zscore_col_name)
        )

df_zscore = df.select("*", *zscore_exprs)

In [ ]:
max_abs_zscore_expr = greatest(*[spark_abs(col(c)) for c in zscore_cols])

df_scored_zscore = df_zscore.withColumn(
    "zscore_outlier_score",
    max_abs_zscore_expr
)


# Anomaly score
df_scored_zscore.select(feature_cols + ["zscore_outlier_score"]).show(5)

In [ ]:
Z_THRESHOLD = 3.5

df_scored_zscore.cache()

initial_count = df_scored_zscore.count()

df_clean = df_scored_zscore.filter(col("zscore_outlier_score") <= Z_THRESHOLD)

# Counting
final_count = df_clean.count()

df_scored_zscore.unpersist()

# percentual
outliers_eliminated_count = initial_count - final_count

percentage_eliminated = (outliers_eliminated_count / initial_count) * 100

In [ ]:
initial_count, final_count, outliers_eliminated_count, percentage_eliminated

In [ ]:
df_final = df_clean

zscore_cols_to_drop = [c for c in df_final.columns if c.startswith('zscore_')]
zscore_cols_to_drop.append("zscore_outlier_score")
zscore_cols_to_drop = list(set(zscore_cols_to_drop)) # Remove duplicates if any

df_final = df_final.drop(*zscore_cols_to_drop)

In [ ]:
# Output
new_output_path = "/content/drive/MyDrive/DDAM project/data/risultato_pca_pulito_finale.csv"

df_final.write.csv(
    new_output_path,
    header=True,
    mode="overwrite",
    sep=","
)

print(f"Cleaned Dataframe (without outliers) saved in:")
print(f"   -> {new_output_path}")